In [1]:
import os
import glob
import warnings
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings("ignore")

# Set the local California data folder
DATA_FOLDER = r"C:\Users\JessicaW\Desktop\IDXExchange\california"

# Explicitly forbidden columns from the task
FORBIDDEN_COLUMNS = [
    "ListPrice",
    "OriginalListPrice"]

# Columns that are typically listing-only and should not be used for modeling
LISTING_ONLY_COLUMNS = [
    "StandardStatus",
    "MlsStatus",
    "ListingContractDate",
    "OnMarketDate",
    "CumulativeDaysOnMarket",
    "DaysOnMarket",
    "PendingTimestamp",
    "PurchaseContractDate",
    "PriceChangeTimestamp",
    "StatusChangeTimestamp"]

# Core columns required by the task
REQUIRED_CORE_COLUMNS = [
    "ClosePrice",
    "CloseDate",
    "PropertyType",
    "PropertySubType",
    "LivingArea",
    "Latitude",
    "Longitude"]

print("Configuration loaded successfully.")
print(f"Data folder: {DATA_FOLDER}")

Configuration loaded successfully.
Data folder: C:\Users\JessicaW\Desktop\IDXExchange\california


In [8]:
# Train / test split
train_df, test_df, test_month = choose_time_split(df)

print("Official test month:", test_month)
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTraining months:")
print(sorted(train_df["CloseMonthPeriod"].dropna().unique()))

print("\nTest month row count:")
print(test_df["CloseMonthPeriod"].value_counts())

# Sanity check for official split rule
assert test_df["CloseMonthPeriod"].nunique() == 1, "Test set must contain exactly one month."
assert test_df["CloseMonthPeriod"].iloc[0] == test_month, "Test set month does not match the official last month."
assert train_df["CloseMonthPeriod"].max() < test_month, "Training data must be strictly earlier than test month."
assert train_df["CloseMonthPeriod"].nunique() >= 6, "Training data must contain at least 6 months before test."
print("Official split sanity check passed.")

Official test month: 2026-02
Train shape: (62535, 96)
Test shape: (8542, 96)

Training months:
[Period('2025-08', 'M'), Period('2025-09', 'M'), Period('2025-10', 'M'), Period('2025-11', 'M'), Period('2025-12', 'M'), Period('2026-01', 'M')]

Test month row count:
CloseMonthPeriod
2026-02    8542
Freq: M, Name: count, dtype: int64
Official split sanity check passed.


# XGBoost

In [16]:
# Train multiple models with log target + recency weights
import time
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

def build_preprocessor_sparse(X):
    numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
    categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median"))])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True))])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features)])
    return preprocessor


def train_and_evaluate_models_with_progress(X_train, y_train, X_test, y_test, sample_weights=None):
    y_train_log = np.log1p(y_train)

    model_dict = {
        "LinearRegression": LinearRegression(),
        "RandomForest": RandomForestRegressor(
            n_estimators=180,
            max_depth=16,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1),
        "XGBoost": XGBRegressor(
            n_estimators=1000,
            learning_rate=0.03,
            max_depth=8,
            min_child_weight=3,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.0,
            reg_lambda=2.0,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1)}

    results = []
    trained_pipelines = {}
    total_models = len(model_dict)

    for i, (model_name, model) in enumerate(model_dict.items(), start=1):
        print(f"\n[{i}/{total_models}] Training {model_name}...")
        start_time = time.time()

        preprocessor = build_preprocessor_sparse(X_train)

        pipe = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", model)])

        if model_name == "LinearRegression":
            pipe.fit(X_train, y_train_log)
        else:
            if sample_weights is not None:
                pipe.fit(X_train, y_train_log, model__sample_weight=sample_weights)
            else:
                pipe.fit(X_train, y_train_log)

        pred_log = pipe.predict(X_test)
        pred_test = np.expm1(pred_log)
        pred_test = np.maximum(pred_test, 1.0)

        y_test_trim, pred_test_trim = trim_test_for_metrics(y_test, pred_test)

        r2 = r2_score(y_test_trim, pred_test_trim)
        med_ape = mdape(y_test_trim, pred_test_trim)

        elapsed = time.time() - start_time

        print(f"{model_name} finished in {elapsed:.2f} seconds.")
        print(f"Trimmed Test R²: {r2:.4f}")
        print(f"Trimmed Test MdAPE: {med_ape:.2f}%")

        results.append({
            "Model": model_name,
            "R2_trimmed_test": round(r2, 4),
            "MdAPE_trimmed_test": round(med_ape, 2),
            "Test_rows_before_trim": len(y_test),
            "Test_rows_after_trim": len(y_test_trim),
            "Train_time_seconds": round(elapsed, 2)})

        trained_pipelines[model_name] = pipe

    results_df = pd.DataFrame(results).sort_values(
        by=["MdAPE_trimmed_test", "R2_trimmed_test"],
        ascending=[True, False]).reset_index(drop=True)

    return results_df, trained_pipelines


results_df, trained_models = train_and_evaluate_models_with_progress(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    sample_weights=train_sample_weights)

print("\nFinal model comparison results:")
display(results_df)


[1/3] Training LinearRegression...
LinearRegression finished in 0.48 seconds.
Trimmed Test R²: 0.3612
Trimmed Test MdAPE: 30.94%

[2/3] Training RandomForest...
RandomForest finished in 71.20 seconds.
Trimmed Test R²: 0.8341
Trimmed Test MdAPE: 9.71%

[3/3] Training XGBoost...
XGBoost finished in 16.97 seconds.
Trimmed Test R²: 0.8602
Trimmed Test MdAPE: 9.81%

Final model comparison results:


,Model,R2_trimmed_test,MdAPE_trimmed_test,Test_rows_before_trim,Test_rows_after_trim,Train_time_seconds
0,RandomForest,0.8341,9.71,8542,8458,71.20
1,XGBoost,0.8602,9.81,8542,8458,16.97
2,LinearRegression,0.3612,30.94,8542,8458,0.48


In [21]:
from pathlib import Path

extra_test_paths = [
    os.path.join(DATA_FOLDER, "CRMLSSold202601_filled.csv") if os.path.exists(os.path.join(DATA_FOLDER, "CRMLSSold202601_filled.csv")) else os.path.join(DATA_FOLDER, "CRMLSSold202601.csv"),
    os.path.join(DATA_FOLDER, "CRMLSSold202602_filled.csv") if os.path.exists(os.path.join(DATA_FOLDER, "CRMLSSold202602_filled.csv")) else os.path.join(DATA_FOLDER, "CRMLSSold202602.csv")]

def load_and_prepare_extra_test_file(file_path):
    """
    Load one extra test CSV and apply the same cleaning / feature engineering
    logic used in the main pipeline.
    """
    temp = pd.read_csv(file_path, low_memory=False).copy()

    # Parse CloseDate
    temp["CloseDate"] = pd.to_datetime(temp["CloseDate"], errors="coerce")

    # Standardize optional boolean flags if present
    temp = standardize_boolean_flags(temp, ["latfilled", "lonfilled"])

    # Apply official property filters
    temp = apply_hard_filters(temp)

    # Remove invalid rows
    temp = remove_invalid_rows(temp)

    # Keep only rows with valid CloseDate
    temp = temp[temp["CloseDate"].notna()].copy()

    # Add same address features as training
    temp = add_light_address_features(temp)

    # Add same engineered features as training
    temp = add_basic_engineered_features(temp)

    return temp


def evaluate_one_extra_month(model, df_month, feature_columns, label):
    """
    Evaluate one external month using the same metric logic:
    trimming is applied only for evaluation metrics.
    """
    temp = df_month.copy()
    temp["ClosePrice"] = pd.to_numeric(temp["ClosePrice"], errors="coerce")
    temp = temp[temp["ClosePrice"].notna()].copy()

    X_month = temp.reindex(columns=feature_columns)
    y_month = temp["ClosePrice"].copy()

    pred_log = model.predict(X_month)
    pred_month = np.expm1(pred_log)
    pred_month = np.maximum(pred_month, 1.0)

    y_trim, pred_trim = trim_test_for_metrics(y_month, pred_month)

    result = {
        "TestLabel": label,
        "RowsBeforeTrim": len(y_month),
        "RowsAfterTrim": len(y_trim),
        "R2": r2_score(y_trim, pred_trim),
        "MdAPE": mdape(y_trim, pred_trim)}
    return result


extra_month_dfs = []
extra_results = []

for path in extra_test_paths:
    if not os.path.exists(path):
        print(f"File not found, skipped: {path}")
        continue
    temp_df = load_and_prepare_extra_test_file(path)
    label = Path(path).stem
    extra_month_dfs.append((label, temp_df))

for label, temp_df in extra_month_dfs:
    result = evaluate_one_extra_month(
        model=best_model,
        df_month=temp_df,
        feature_columns=X_train.columns.tolist(),
        label=label)
    extra_results.append(result)

if len(extra_month_dfs) > 0:
    combined_extra_df = pd.concat([df for _, df in extra_month_dfs], axis=0, ignore_index=True)

    combined_result = evaluate_one_extra_month(
        model=best_model,
        df_month=combined_extra_df,
        feature_columns=X_train.columns.tolist(),
        label="CRMLSSold202601_202602_combined")
    extra_results.append(combined_result)

extra_results_df = pd.DataFrame(extra_results).sort_values("TestLabel").reset_index(drop=True)

print("Extra out-of-time validation results:")
display(extra_results_df)

Created light address features from UnparsedAddress.
Created light address features from UnparsedAddress.
Extra out-of-time validation results:


,TestLabel,RowsBeforeTrim,RowsAfterTrim,R2,MdAPE
0,CRMLSSold202601,7473,7397,0.928125,6.756829
1,CRMLSSold202601_202602_combined,16015,15856,0.876268,8.212810
2,CRMLSSold202602,8542,8458,0.834075,9.710530
